In [1]:
# Setup

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print(PROJECT_ROOT)

g:\My Drive\Graduação e Pós\USP\MBA USP IA e Big Data\TCC\port_leadtime_prediction


In [2]:
# Imports

import pandas as pd

from src.processing.targets import (
    filter_eligible_port_calls,
    create_duration_targets,
    create_duration_targets_in_days,
    create_log_targets,
    create_target_severity_flags,
    build_target_summary,
)

In [3]:
# Loading validated master table

qc_path = PROJECT_ROOT / "data" / "interim" / "master_port_calls_qc.parquet"

df_qc = pd.read_parquet(qc_path)

print(df_qc.shape)
df_qc.head()

(146037, 37)


,port_call_id,port,port_name,imo,vessel_id,vessel_name,operation_type,source_port,source_port_name,destination_port,...,tmp_post_operation_h,tmp_total_port_stay_h,flag_negative_tmp_wait_for_berthing_h,flag_negative_tmp_operation_h,flag_negative_tmp_post_operation_h,flag_negative_tmp_total_port_stay_h,flag_wait_too_long,flag_operation_too_long,flag_total_too_long,eligible_for_eda
0,561202021,BR052001,TERMINAL FLUVIAL DE JURUTI,9620310,<NA>,WHITE WHALE,Carga,BRAMW001,TERMINAL DA ALUMAR - SÃO LUIZ - MA,BR052,...,0.0,25.683333,False,False,False,False,False,False,False,True
1,4172022,BR052001,TERMINAL FLUVIAL DE JURUTI,9304241,<NA>,GOOD HOPE MAX,Carga,BR052,JURUTI,BRAMW001,...,0.0,42.000000,False,False,False,False,False,False,False,True
2,7012022,BR052001,TERMINAL FLUVIAL DE JURUTI,9628922,<NA>,AMBERJACK,Carga,BRAMW001,TERMINAL DA ALUMAR - SÃO LUIZ - MA,BRAMW001,...,0.0,28.600000,False,False,False,False,False,False,False,True
3,12292022,BR052001,TERMINAL FLUVIAL DE JURUTI,9473339,<NA>,JURUTI,Carga,BRAMW001,TERMINAL DA ALUMAR - SÃO LUIZ - MA,BR052,...,0.0,33.283333,False,False,False,False,False,False,False,True
4,19472022,BR052001,TERMINAL FLUVIAL DE JURUTI,9620310,<NA>,WHITE WHALE,Carga,BR052,JURUTI,BR052,...,0.0,24.800000,False,False,False,False,False,False,False,True


In [4]:
# Filtering eligible port calls for EDA and target creation

df_target = filter_eligible_port_calls(df_qc)

print(df_target.shape)
df_target["eligible_for_eda"].value_counts(dropna=False)

(141895, 37)


eligible_for_eda
True    141895
Name: count, dtype: int64

In [5]:
# Creating the final targets

df_target = create_duration_targets(df_target)
df_target = create_duration_targets_in_days(df_target)
df_target = create_log_targets(df_target)
df_target = create_target_severity_flags(df_target)

print(df_target.shape)
df_target.head()

(141895, 55)


,port_call_id,port,port_name,imo,vessel_id,vessel_name,operation_type,source_port,source_port_name,destination_port,...,log_t_wait_for_berthing_h,log_t_operation_h,log_t_post_operation_h,log_t_total_port_stay_h,t_wait_for_berthing_h_high,t_wait_for_berthing_h_extreme,t_operation_h_high,t_operation_h_extreme,t_total_port_stay_h_high,t_total_port_stay_h_extreme
0,561202021,BR052001,TERMINAL FLUVIAL DE JURUTI,9620310,<NA>,WHITE WHALE,Carga,BRAMW001,TERMINAL DA ALUMAR - SÃO LUIZ - MA,BR052,...,0.0,3.284039,0.0,3.284039,False,False,False,False,False,False
1,4172022,BR052001,TERMINAL FLUVIAL DE JURUTI,9304241,<NA>,GOOD HOPE MAX,Carga,BR052,JURUTI,BRAMW001,...,0.0,3.761200,0.0,3.761200,False,False,False,False,False,False
2,7012022,BR052001,TERMINAL FLUVIAL DE JURUTI,9628922,<NA>,AMBERJACK,Carga,BRAMW001,TERMINAL DA ALUMAR - SÃO LUIZ - MA,BRAMW001,...,0.0,3.387774,0.0,3.387774,False,False,False,False,False,False
3,12292022,BR052001,TERMINAL FLUVIAL DE JURUTI,9473339,<NA>,JURUTI,Carga,BRAMW001,TERMINAL DA ALUMAR - SÃO LUIZ - MA,BR052,...,0.0,3.534659,0.0,3.534659,False,False,False,False,False,False
4,19472022,BR052001,TERMINAL FLUVIAL DE JURUTI,9620310,<NA>,WHITE WHALE,Carga,BR052,JURUTI,BR052,...,0.0,3.250374,0.0,3.250374,False,False,False,False,False,False


In [6]:
# Checking only the main targets

target_cols = [
    "t_wait_for_berthing_h",
    "t_operation_h",
    "t_post_operation_h",
    "t_total_port_stay_h",
]

df_target[target_cols].head(10)

,t_wait_for_berthing_h,t_operation_h,t_post_operation_h,t_total_port_stay_h
0,0.0,25.683333,0.0,25.683333
1,0.0,42.000000,0.0,42.000000
2,0.0,28.600000,0.0,28.600000
3,0.0,33.283333,0.0,33.283333
4,0.0,24.800000,0.0,24.800000
5,0.0,27.100000,0.0,27.100000
6,0.0,51.300000,0.0,51.300000
7,0.0,29.716667,0.0,29.716667
8,0.0,30.400000,0.0,30.400000
9,0.0,27.516667,0.0,27.516667


In [7]:
# Checking if there are negative values in the targets - expected to be zero.

(df_target[target_cols] < 0).sum()

t_wait_for_berthing_h    0
t_operation_h            0
t_post_operation_h       0
t_total_port_stay_h      0
dtype: int64

In [8]:
# Statistical summary of the targets

target_summary = build_target_summary(df_target)
target_summary

,target,count,mean,median,std,min,p25,p75,p90,p95,max
0,t_wait_for_berthing_h,141895,10.323023,0.000000,63.834458,0.000000,0.000000,0.000000,6.900000,40.626667,3000.950000
1,t_operation_h,141895,99.676047,35.416667,791.800350,0.016667,17.500000,74.333333,153.566667,254.610000,77638.716667
2,t_post_operation_h,141895,0.830300,0.000000,19.892532,0.000000,0.000000,0.000000,0.000000,0.000000,3958.166667
3,t_total_port_stay_h,141895,110.829370,39.350000,795.278469,0.016667,19.083333,85.250000,183.550000,313.583333,77638.716667


In [9]:
# Checking the severity flags

severity_cols = [
    "t_wait_for_berthing_h_high",
    "t_wait_for_berthing_h_extreme",
    "t_operation_h_high",
    "t_operation_h_extreme",
    "t_total_port_stay_h_high",
    "t_total_port_stay_h_extreme",
]

df_target[severity_cols].mean().sort_values(ascending=False)

t_total_port_stay_h_high         0.249896
t_operation_h_high               0.249833
t_wait_for_berthing_h_high       0.130864
t_wait_for_berthing_h_extreme    0.099996
t_operation_h_extreme            0.099989
t_total_port_stay_h_extreme      0.099989
dtype: float64

In [10]:
# Saving the results

interim_dir = PROJECT_ROOT / "data" / "interim"
tables_dir = PROJECT_ROOT / "outputs" / "tables"

interim_dir.mkdir(parents=True, exist_ok=True)
tables_dir.mkdir(parents=True, exist_ok=True)

target_base_path = interim_dir / "target_base.parquet"
target_summary_path = tables_dir / "target_summary.csv"

df_target.to_parquet(target_base_path, index=False)
target_summary.to_csv(target_summary_path, index=False)

print(f"Target base saved to: {target_base_path}")
print(f"Target summary saved to: {target_summary_path}")

Target base saved to: g:\My Drive\Graduação e Pós\USP\MBA USP IA e Big Data\TCC\port_leadtime_prediction\data\interim\target_base.parquet
Target summary saved to: g:\My Drive\Graduação e Pós\USP\MBA USP IA e Big Data\TCC\port_leadtime_prediction\outputs\tables\target_summary.csv
